# Prepare our flight and weather data

Run each cell from top to bottom. We will look at the original files, keep the data we need, and save one combined CSV.

We are using **2024 departures from Knoxville, Nashville, Memphis, and Chattanooga**.


## 1. Choose the months
Use `list(range(1, 13))` for the full year, or `[1]` for a quicker January run.

In [8]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display

# Find our project folder and load its supporting code.
ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
             if (folder / "Prepare_Data.ipynb").exists()
             and (folder / "data_helpers.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the FlightDelaysAndWeather folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import data_helpers
importlib.reload(data_helpers)
from data_helpers import ensure_data, load_flights, load_weather, match_weather

YEAR = 2024
MONTHS = list(range(1, 13))  # Change to [1] to try January first.
assert MONTHS and all(isinstance(m, int) and 1 <= m <= 12 for m in MONTHS)
assert len(MONTHS) == len(set(MONTHS))
print("Year:", YEAR, "| Months:", MONTHS)


Year: 2024 | Months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


## 2. Get the files
Missing files download automatically. Files we already have are reused. The first full-year download may take a few minutes.

In [9]:
flight_paths, weather_paths, sources = ensure_data(ROOT, YEAR, MONTHS)
print(f"Ready: {len(flight_paths)} flight files and {len(weather_paths)} weather files.")


Ready: 12 flight files and 4 weather files.


## 3. Look at the raw data
Each flight row is one scheduled flight. Each weather row is a reading from an airport weather station.
The weather values contain codes; we will turn those into readable numbers in step 5.

In [10]:
print("Flight sample:")
raw_flights = pd.read_csv(flight_paths[0], nrows=5)
display(raw_flights[["FlightDate", "Origin", "Dest", "CRSDepTime", "DepDelay", "Cancelled"]])

print("Knoxville weather sample:")
raw_weather = pd.read_csv(weather_paths["TYS"], nrows=5, dtype="string")
display(raw_weather[["DATE", "TMP", "WND", "VIS"]])


Flight sample:


,FlightDate,Origin,Dest,CRSDepTime,DepDelay,Cancelled
0,2024-01-08,LGA,OMA,856,-5.0,0.0
1,2024-01-09,LGA,OMA,856,-5.0,0.0
2,2024-01-10,LGA,OMA,856,-6.0,0.0
3,2024-01-11,LGA,OMA,856,23.0,0.0
4,2024-01-12,LGA,OMA,856,-5.0,0.0


Knoxville weather sample:


,DATE,TMP,WND,VIS
0,2024-01-01T00:00:00,"+0100,1","210,1,N,0062,1","016000,1,9,9"
1,2024-01-01T00:53:00,"+0083,5","230,5,N,0051,5","016093,5,N,5"
2,2024-01-01T01:53:00,"+0089,5","220,5,N,0062,5","016093,5,N,5"
3,2024-01-01T02:53:00,"+0089,5","210,5,N,0062,5","016093,5,N,5"
4,2024-01-01T03:00:00,"+0089,1","210,1,N,0062,1","016000,1,9,9"


## 4. Keep our flights
Keep departures from TYS, BNA, MEM, and CHA for the selected months. Keep cancellations too.
Times are put on the same clock so we can match them with weather later.

In [11]:
flights, flight_counts, duplicates_removed = load_flights(flight_paths, YEAR, MONTHS)
print(f"Read {flight_counts.raw_rows.sum():,} nationwide flight records.")
print(f"Kept {len(flights):,} flights from our four airports.")
print(f"Removed {duplicates_removed} exact duplicate records.")
display(flights.groupby("Origin").agg(
    flights=("flight_id", "size"), cancellations=("Cancelled", "sum")))
display(flights[["FlightDate", "Origin", "Dest", "DepDelay", "Cancelled"]].head())


Read 7,079,061 nationwide flight records.
Kept 145,978 flights from our four airports.
Removed 0 exact duplicate records.


,flights,cancellations
Origin,,
BNA,100220,1273.0
CHA,6853,85.0
MEM,22624,306.0
TYS,16281,257.0


,FlightDate,Origin,Dest,DepDelay,Cancelled
0,2024-01-13,BNA,LGA,-5.0,0.0
1,2024-01-20,BNA,LGA,-3.0,0.0
2,2024-01-27,BNA,LGA,-5.0,0.0
3,2024-01-02,MEM,LGA,-4.0,0.0
4,2024-01-03,MEM,LGA,-5.0,0.0


## 5. Clean the weather
Keep airport weather reports and turn the codes into temperature, wind speed, visibility, and cloud ceiling.
Missing or questionable readings stay blank. Blank does not mean zero.
Weather flags use 1 when a condition was reported, 0 when usable codes did not report it, and blank when unknown. One-hour precipitation is in millimeters, with a separate flag for trace amounts.

In [12]:
weather, weather_counts = load_weather(weather_paths, YEAR, MONTHS)
print(f"Read {weather_counts.raw_rows.sum():,} weather reports.")
print(f"Kept {len(weather):,} reports for matching.")
display(weather[["Origin", "weather_time_utc", "TMP_raw", "temperature_c",
                 "wind_speed_mps", "visibility_m"]].head())
print("Examples with usable condition codes (up to five reports):")
display(weather.loc[weather["condition_codes_available"],
                    ["Origin", "rain_reported", "snow_reported", "fog_reported",
                     "thunderstorm_reported", "precipitation_1hr_mm"]].head())
print("Missing readings:")
display(weather[["temperature_c", "wind_speed_mps", "visibility_m"]]
        .isna().sum().to_frame("missing"))


/home/ksaravan/FlightDelaysAndWeather/data_helpers.py:184: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return add_weather_categories(pd.concat(pieces, ignore_index=True)), pd.DataFrame(inventory)


Read 55,399 weather reports.
Kept 42,450 reports for matching.


,Origin,weather_time_utc,TMP_raw,temperature_c,wind_speed_mps,visibility_m
0,TYS,2024-01-01 00:53:00+00:00,"+0083,5",8.3,5.1,16093.0
1,TYS,2024-01-01 01:53:00+00:00,"+0089,5",8.9,6.2,16093.0
2,TYS,2024-01-01 02:53:00+00:00,"+0089,5",8.9,6.2,16093.0
3,TYS,2024-01-01 03:53:00+00:00,"+0083,5",8.3,5.7,16093.0
4,TYS,2024-01-01 04:53:00+00:00,"+0072,5",7.2,4.6,16093.0


Examples with usable condition codes (up to five reports):


,Origin,rain_reported,snow_reported,fog_reported,thunderstorm_reported,precipitation_1hr_mm
80,TYS,1,0,0,0,<NA>
81,TYS,1,0,0,0,<NA>
82,TYS,1,0,0,0,<NA>
143,TYS,1,0,0,0,1.0
144,TYS,1,0,0,0,0.8


Missing readings:


,missing
temperature_c,85
wind_speed_mps,38
visibility_m,72


## 6. Match each flight with weather
Use the most recent report from the **same airport**, at or before scheduled departure, within 90 minutes.
Flights without a match stay in the table with blank weather. Each row should still represent one flight.

In [13]:
combined = match_weather(flights, weather, tolerance_minutes=90)
summary = combined.groupby("Origin").agg(
    flights=("flight_id", "size"), weather_matches=("weather_matched", "sum"))
summary["no_weather_match"] = summary["flights"] - summary["weather_matches"]
display(summary)
print(f"Of {len(combined):,} scheduled flights, {combined.weather_matched.sum():,} matched a weather report.")
print(f"{combined.condition_codes_available.eq(True).sum():,} flights had usable condition codes.")
print(f"{combined.precipitation_1hr_mm.count():,} flights had a measured one-hour precipitation amount.")
display(combined[["FlightDate", "Origin", "scheduled_departure_utc",
                  "weather_time_utc", "weather_age_minutes", "DepDelay", "Cancelled"]].head())


,flights,weather_matches,no_weather_match
Origin,,,
BNA,100220,100139,81
CHA,6853,6840,13
MEM,22624,22548,76
TYS,16281,16201,80


Of 145,978 scheduled flights, 145,728 matched a weather report.
12,795 flights had usable condition codes.
129,022 flights had a measured one-hour precipitation amount.


,FlightDate,Origin,scheduled_departure_utc,weather_time_utc,weather_age_minutes,DepDelay,Cancelled
0,2024-01-13,BNA,2024-01-13 20:00:00+00:00,2024-01-13 19:53:00+00:00,7.0,-5.0,0.0
1,2024-01-20,BNA,2024-01-20 20:00:00+00:00,2024-01-20 19:53:00+00:00,7.0,-3.0,0.0
2,2024-01-27,BNA,2024-01-27 20:00:00+00:00,2024-01-27 19:53:00+00:00,7.0,-5.0,0.0
3,2024-01-02,MEM,2024-01-02 18:55:00+00:00,2024-01-02 18:54:00+00:00,1.0,-4.0,0.0
4,2024-01-03,MEM,2024-01-03 18:55:00+00:00,2024-01-03 18:54:00+00:00,1.0,-5.0,0.0


## 7. Save the combined CSV
Check that no flights or cancellations were lost, then save the result under `data/processed/`.
January and full-year runs have different filenames. Original downloads stay unchanged.

In [14]:
assert len(combined) == len(flights)
assert combined["flight_id"].is_unique
assert combined["Cancelled"].sum() == flights["Cancelled"].sum()
matched = combined["weather_matched"]
assert combined.loc[matched, "weather_age_minutes"].between(0, 90).all()
print("Checks passed: flight counts, cancellations, and weather timing.")
print("Possible duplicate flights to review:", int(combined["possible_duplicate_flight"].sum()))
print("Matched reports with no usable core readings:",
      int((matched & combined["valid_weather_measurements"].eq(0)).sum()))

output_dir = ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
period = "full_year" if sorted(MONTHS) == list(range(1, 13)) else "months_" + "_".join(f"{m:02}" for m in sorted(MONTHS))
output_path = output_dir / f"flights_weather_{YEAR}_{period}.csv"
combined.to_csv(output_path, index=False)
assert len(pd.read_csv(output_path, usecols=["flight_id"])) == len(combined)
print(f"Saved {len(combined):,} flights to {output_path.relative_to(ROOT)}")


Checks passed: flight counts, cancellations, and weather timing.
Possible duplicate flights to review: 0
Matched reports with no usable core readings: 6
Saved 145,978 flights to data/processed/flights_weather_2024_full_year.csv


## What comes next?
Use the combined CSV to compare delay and cancellation rates across airports and weather conditions. These matches show weather near a flight's departure; they do not prove why it was delayed.
